Random Forest

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/audio_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # -------------------------
    # Keep numeric columns only
    # -------------------------
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # -------------------------
    # Drop missing values
    # -------------------------
    df_phase = df_phase.dropna()

    # -------------------------
    # Features and Labels
    # -------------------------
    X = df_phase.drop("label", axis=1).values
    y = df_phase["label"].values

    print("Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    # =====================================================
    # FEATURE SCALING
    # =====================================================
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # =====================================================
    # RANDOM FOREST MODEL
    # =====================================================
    model = RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # =====================================================
    # PREDICTIONS
    # =====================================================
    THRESHOLD = 0.40

    y_prob = model.predict_proba(X_test)[:, 1]

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # =====================================================
    # ROC AUC
    # =====================================================
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return model, roc

# =========================================================
# 4️⃣ FILTER PHASE-WISE DATA
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 771)

Available Phases:
['phase1' 'phase2' 'phase3']

Total Features: 768

TRAINING FOR PHASE 1
Shape: (141, 769)
Class Distribution: [99 42]

Predicted Distribution:
[24  5]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.85      0.77        20
           1       0.40      0.22      0.29         9

    accuracy                           0.66        29
   macro avg       0.55      0.54      0.53        29
weighted avg       0.61      0.66      0.62        29

ROC-AUC Score: 0.5139

Confusion Matrix:
[[17  3]
 [ 7  2]]

TRAINING FOR PHASE 2
Shape: (141, 769)
Class Distribution: [99 42]

Predicted Distribution:
[26  3]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.95      0.83        20
           1       0.67      0.22      0.33         9

    accuracy            

Random Forest 5 folds

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/audio_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION (5-FOLD RF)
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # -------------------------
    # Keep numeric columns only
    # -------------------------
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # -------------------------
    # Drop missing values
    # -------------------------
    df_phase = df_phase.dropna()

    # -------------------------
    # Features and Labels
    # -------------------------
    X = df_phase.drop("label", axis=1).values
    y = df_phase["label"].values

    print("Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # RANDOM FOREST MODEL
        # =================================================
        model = RandomForestClassifier(
            n_estimators=400,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        THRESHOLD = 0.40

        y_prob = model.predict_proba(X_test)[:, 1]

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER PHASE-WISE DATA
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

TRAINING FOR PHASE 1
Shape: (141, 132)
Class Distribution: [99 42]

----- Fold 1 -----
Fold ROC-AUC: 0.6278

----- Fold 2 -----
Fold ROC-AUC: 0.8094

----- Fold 3 -----
Fold ROC-AUC: 0.6531

----- Fold 4 -----
Fold ROC-AUC: 0.7188

----- Fold 5 -----
Fold ROC-AUC: 0.6579

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.6278 0.8094 0.6531 0.7188 0.6579]

Mean Fold ROC-AUC:
0.6934

Overall ROC-AUC:
0.6762

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.87      0.81        99
           1       0.52      0.33      0.41        42

    accuracy                           0.71       141
   macro avg       0.64      0.60      0.61       141
weighted avg       0.68      0.71      0.69       141


Confusion Matrix:
[[86 13]
 [28 14]]

TRAINING FOR PHASE 2
Shape: (141, 132)
Class D

Random Forest 10 folds

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/audio_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION (5-FOLD RF)
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # -------------------------
    # Keep numeric columns only
    # -------------------------
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # -------------------------
    # Drop missing values
    # -------------------------
    df_phase = df_phase.dropna()

    # -------------------------
    # Features and Labels
    # -------------------------
    X = df_phase.drop("label", axis=1).values
    y = df_phase["label"].values

    print("Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # RANDOM FOREST MODEL
        # =================================================
        model = RandomForestClassifier(
            n_estimators=400,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        THRESHOLD = 0.40

        y_prob = model.predict_proba(X_test)[:, 1]

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER PHASE-WISE DATA
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

TRAINING FOR PHASE 1
Shape: (141, 132)
Class Distribution: [99 42]

----- Fold 1 -----
Fold ROC-AUC: 0.64

----- Fold 2 -----
Fold ROC-AUC: 0.5

----- Fold 3 -----
Fold ROC-AUC: 0.6625

----- Fold 4 -----
Fold ROC-AUC: 0.8

----- Fold 5 -----
Fold ROC-AUC: 0.7875

----- Fold 6 -----
Fold ROC-AUC: 0.65

----- Fold 7 -----
Fold ROC-AUC: 0.6875

----- Fold 8 -----
Fold ROC-AUC: 0.7875

----- Fold 9 -----
Fold ROC-AUC: 0.7

----- Fold 10 -----
Fold ROC-AUC: 0.5556

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.64   0.5    0.6625 0.8    0.7875 0.65   0.6875 0.7875 0.7    0.5556]

Mean Fold ROC-AUC:
0.6771

Overall ROC-AUC:
0.6751

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.88      0.81        99
           1       0.52      0.31      0.39        42

    accuracy        

SVM

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/audio_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # -------------------------
    # Keep numeric columns only
    # -------------------------
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # -------------------------
    # Drop missing values
    # -------------------------
    df_phase = df_phase.dropna()

    # -------------------------
    # Features and Labels
    # -------------------------
    X = df_phase.drop("label", axis=1).values
    y = df_phase["label"].values

    print("Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # =====================================================
    # FEATURE SCALING
    # =====================================================
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # =====================================================
    # SVM MODEL
    # =====================================================
    model = SVC(

        # RBF kernel
        kernel="rbf",

        # Needed for ROC-AUC
        probability=True,

        # Handle imbalance
        class_weight="balanced",

        # Tuned parameters
        C=2,

        gamma="scale",

        random_state=42
    )

    # =====================================================
    # TRAIN MODEL
    # =====================================================
    model.fit(X_train, y_train)

    # =====================================================
    # PREDICT
    # =====================================================
    THRESHOLD = 0.40

    y_prob = model.predict_proba(X_test)[:, 1]

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # =====================================================
    # ROC-AUC
    # =====================================================
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return model, roc

# =========================================================
# 4️⃣ FILTER PHASE-WISE DATA
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

TRAINING FOR PHASE 1
Shape: (141, 132)
Class Distribution: [99 42]

Predicted Distribution:
[25  4]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.85      0.76        20
           1       0.25      0.11      0.15         9

    accuracy                           0.62        29
   macro avg       0.47      0.48      0.45        29
weighted avg       0.55      0.62      0.57        29

ROC-AUC Score: 0.5444

Confusion Matrix:
[[17  3]
 [ 8  1]]

TRAINING FOR PHASE 2
Shape: (141, 132)
Class Distribution: [99 42]

Predicted Distribution:
[29]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.69      1.00      0.82        20
           1       0.00      0.00      0.00 

SVM 5 folds

In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/audio_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # KEEP NUMERIC ONLY
    # =====================================================
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase.drop("label", axis=1)
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    selector = VarianceThreshold(
        threshold=0.002
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # TUNED SVM MODEL
        # =================================================
        model = SVC(

            # Nonlinear separation
            kernel="rbf",

            # Needed for ROC-AUC
            probability=True,

            # Handle imbalance
            class_weight="balanced",

            # Stronger fitting
            C=3,

            # Smoother boundary
            gamma=0.0005,

            random_state=42
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        THRESHOLD = 0.40

        y_prob = model.predict_proba(X_test)[:, 1]

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER PHASE-WISE DATA
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

TRAINING FOR PHASE 1
Original Shape: (141, 132)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 105)

----- Fold 1 -----
Fold ROC-AUC: 0.5722

----- Fold 2 -----
Fold ROC-AUC: 0.675

----- Fold 3 -----
Fold ROC-AUC: 0.6813

----- Fold 4 -----
Fold ROC-AUC: 0.7437

----- Fold 5 -----
Fold ROC-AUC: 0.6959

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.5722 0.675  0.6812 0.7437 0.6959]

Mean Fold ROC-AUC:
0.6736

Overall ROC-AUC:
0.67

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.98      0.83        99
           1       0.60      0.07      0.13        42

    accuracy                           0.71       141
   macro avg       0.66      0.53      0.48       141
weighted avg       0.68      0.71      0.62       141


Confusion Matrix:
[[97  2]
 [39  3]

SVM 10 folds

In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/audio_features_phasewise.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # KEEP NUMERIC ONLY
    # =====================================================
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase.drop("label", axis=1)
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    selector = VarianceThreshold(
        threshold=0.002
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # TUNED SVM MODEL
        # =================================================
        model = SVC(

            # Nonlinear separation
            kernel="rbf",

            # Needed for ROC-AUC
            probability=True,

            # Handle imbalance
            class_weight="balanced",

            # Stronger fitting
            C=3,

            # Smoother boundary
            gamma=0.0005,

            random_state=42
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        THRESHOLD = 0.40

        y_prob = model.predict_proba(X_test)[:, 1]

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER PHASE-WISE DATA
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

TRAINING FOR PHASE 1
Original Shape: (141, 132)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 105)

----- Fold 1 -----
Fold ROC-AUC: 0.62

----- Fold 2 -----
Fold ROC-AUC: 0.5

----- Fold 3 -----
Fold ROC-AUC: 0.65

----- Fold 4 -----
Fold ROC-AUC: 0.7

----- Fold 5 -----
Fold ROC-AUC: 0.15

----- Fold 6 -----
Fold ROC-AUC: 0.675

----- Fold 7 -----
Fold ROC-AUC: 0.725

----- Fold 8 -----
Fold ROC-AUC: 0.725

----- Fold 9 -----
Fold ROC-AUC: 0.525

----- Fold 10 -----
Fold ROC-AUC: 0.5778

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.62   0.5    0.65   0.7    0.15   0.675  0.725  0.725  0.525  0.5778]

Mean Fold ROC-AUC:
0.5848

Overall ROC-AUC:
0.615

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.96      0.82        99
           1       0.43    